# Estrutura da Página HTML para RPA

## Link (GitHub Pages)

https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n

## Código HTML

```html
<!doctype html>
<html lang="pt-br">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>Clientes</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 24px; }
    table { border-collapse: collapse; width: 100%; max-width: 900px; }
    th, td { border: 1px solid #ddd; padding: 10px; text-align: left; }
    th { background: #f5f5f5; }
  </style>
</head>
<body>
  <h1>Lista de Clientes</h1>

  <table id="clientes">
    <thead>
      <tr>
        <th>Nome</th>
        <th>Email</th>
        <th>Saldo</th>
        <th>Perfil</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td class="nome">Ana Silva</td>
        <td class="email">ana@email.com</td>
        <td class="saldo">R$ 12.500,00</td>
        <td class="perfil">Conservador</td>
      </tr>
      <tr>
        <td class="nome">Bruno Lima</td>
        <td class="email">bruno@email.com</td>
        <td class="saldo">R$ 3.200,00</td>
        <td class="perfil">Moderado</td>
      </tr>
      <tr>
        <td class="nome">Carla Souza</td>
        <td class="email">carla@email.com</td>
        <td class="saldo">R$ 28.900,00</td>
        <td class="perfil">Arrojado</td>
      </tr>
    </tbody>
  </table>
</body>
</html>
```

# Dependências (Python)

In [8]:
!pip install requests beautifulsoup4

# Código-fonte RPA (Pyhton)

In [ ]:
import os
import re

import requests
from bs4 import BeautifulSoup

URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"

# No Colab: Secrets (icone da chave, a esquerda) ou os.environ direto.
N8N_WEBHOOK = os.environ.get("N8N_WEBHOOK_URL", "https://SEU_N8N/webhook/dio-investimentos")
RPA_TOKEN = os.environ.get("RPA_TOKEN", "")


def saldo_para_numero(texto):
    """'R$ 12.500,00' -> 12500.0

    Em pt-BR o ponto e separador de milhar e a virgula e decimal, ao contrario do
    que float() espera. Sem esta conversao, 'R$ 850,00' seria lido como 85000.0 e
    'R$ 12.500,00' como 12.5 -- e a comparacao com o investimento minimo de cada
    produto, que o workflow faz a seguir, daria recomendacoes erradas.
    """
    limpo = re.sub(r"[^\d,.]", "", texto).replace(".", "").replace(",", ".")
    return float(limpo)


# 1) Baixar a pagina
html = requests.get(URL_PAGINA, timeout=30).text

# 2) Ler a tabela de clientes
soup = BeautifulSoup(html, "html.parser")
linhas = soup.select("#clientes tbody tr")
if not linhas:
    raise RuntimeError("Nenhuma linha em '#clientes tbody tr'. A pagina mudou de estrutura?")

clientes = []

for linha in linhas:
    colunas = linha.find_all("td")
    saldo_texto = colunas[2].get_text(strip=True)

    cliente = {
        "nome":  colunas[0].get_text(strip=True),
        "email": colunas[1].get_text(strip=True),
        "saldo_texto": saldo_texto,
        "saldo": saldo_para_numero(saldo_texto),
        "perfil": colunas[3].get_text(strip=True),
    }

    clientes.append(cliente)

# 3) Mostrar o resultado
print(f"{len(clientes)} clientes encontrados:")
for cliente in clientes:
    print(f"  {cliente['nome']:<18} {cliente['perfil']:<12} {cliente['saldo']:>10.2f}")

# 4) Enviar para o Webhook do n8n
#    usar_ia=False -> mensagens por template (MVP)
#    usar_ia=True  -> mensagens geradas pelo modelo de linguagem
if not RPA_TOKEN:
    print("\nRPA_TOKEN nao definido: nada enviado. Define o token e corre outra vez.")
else:
    resposta = requests.post(
        N8N_WEBHOOK,
        json={"clientes": clientes, "usar_ia": False},
        headers={"X-RPA-Token": RPA_TOKEN},
        timeout=300,
    )
    resposta.raise_for_status()

    print(f"\nEnviado. O n8n devolveu {len(resposta.json())} mensagens:\n")
    for item in resposta.json():
        print(f"--- {item['nome']} ({item['perfil']}) -> {item['produto_recomendado']}")
        print(f"{item['mensagem']}\n")
